# In this file, we test the trained unet model on the 2p_5p NFI Dataset. Then we compare the metrics of the model compared to human specialists.

In [1]:
%cd /Users/amarmesic/Documents/tudelft/thesis/DNANet

/Users/amarmesic/Documents/tudelft/thesis/DNANet


/Users/amarmesic/miniconda3/envs/dnanet/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
from DNAnet.data.data_models.hid_dataset import HIDDataset
from DNAnet.evaluation import *
from DNAnet.evaluation.segmentation.allele_metrics import *

from DNAnet.evaluation.visualizations import plot_profile
from config_io import load_model


/Users/amarmesic/miniconda3/envs/dnanet/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
hid_dataset = HIDDataset(
  root="resources/data/2p_5p_Dataset_NFI/Raw data .HID files",
  panel="resources/data/SGPanel_PPF6C_SPOOR.xml",
  annotations_path="resources/data/2p_5p_Dataset_NFI/.txt bestanden 2024 naam",
  hid_to_annotations_path="resources/data/2p_5p_Dataset_NFI/2p_5p_hid_to_annotation.csv",
  limit=30,
  ground_truth_as_annotations=True
)

unet_model = load_model("config/models/unet.yaml")
unet_model.load("resources/model/current_best_unet/")

2025-06-07 10:19:50 INFO     Loading data from file
2025-06-07 10:19:50 INFO     Walking directory to retrieve all hid files...
Processing folders: 100%|██████████| 21/21 [00:00<00:00, 5076.09it/s]
2025-06-07 10:19:50 INFO     Found 350 HIDs
2025-06-07 10:19:50 INFO     Found 0 HIDs without ladder
2025-06-07 10:19:50 INFO     Removed 0 HIDs without annotation
2025-06-07 10:19:50 INFO     Limiting dataset to 30
Loading data from resources/data/2p_5p_Dataset_NFI/Raw data .HID files:   0%|          | 0/30 [00:00<?, ?it/s]2025-06-07 10:19:50 WARNING  Could not load true alleles for resources/data/2p_5p_Dataset_NFI/Raw data .HID files/Mixture dataset 6/Inj1 2017-05-16-15-20-54-079/ladder_G03_21.hid: Cannot load donor alleles for non-RD sample. Found file name ladder_G03_21
Could not load true alleles for resources/data/2p_5p_Dataset_NFI/Raw data .HID files/Mixture dataset 6/Inj1 2017-05-16-15-20-54-079/ladder_G03_21.hid: Cannot load donor alleles for non-RD sample. Found file name ladder_G0

In [5]:
predictions = unet_model.predict_batch(hid_dataset)

2025-06-07 10:23:30 INFO     Calling alleles from predicted segmentation...


In [6]:
predictions[0].meta['called_alleles'][::3]

[Marker(dye_row=0, name='D3S1358', alleles=[Allele(name='16', base_pair=None, left_bin=None, right_bin=None, height=2706), Allele(name='17', base_pair=None, left_bin=None, right_bin=None, height=843), Allele(name='15', base_pair=None, left_bin=None, right_bin=None, height=4721)]),
 Marker(dye_row=0, name='D10S1248', alleles=[Allele(name='16', base_pair=None, left_bin=None, right_bin=None, height=2476), Allele(name='13', base_pair=None, left_bin=None, right_bin=None, height=3041), Allele(name='14', base_pair=None, left_bin=None, right_bin=None, height=530)]),
 Marker(dye_row=1, name='D16S539', alleles=[Allele(name='13', base_pair=None, left_bin=None, right_bin=None, height=10936), Allele(name='11', base_pair=None, left_bin=None, right_bin=None, height=393)]),
 Marker(dye_row=1, name='CSF1PO', alleles=[Allele(name='12', base_pair=None, left_bin=None, right_bin=None, height=4794), Allele(name='11', base_pair=None, left_bin=None, right_bin=None, height=848)]),
 Marker(dye_row=2, name='vWA'

In [7]:
# plot_profile(hid_dataset, predictions)

## Prediction Metrics

In [9]:
print('pixel u-net f1: ', pixel_f1_score(hid_dataset, predictions))
print('pixel u-net precision: ', pixel_precision(hid_dataset, predictions))
print('pixel u-net recall: ', pixel_recall(hid_dataset, predictions))


print('allele u-net f1: ', allele_f1_score(hid_dataset, predictions))
print('allele u-net precision: ', allele_precision(hid_dataset, predictions))
print('allele u-net recall: ', allele_recall(hid_dataset, predictions))


pixel u-net f1:  0.1910284944045641
pixel u-net precision:  0.9970549738219895
pixel u-net recall:  0.10563355867568036
allele u-net f1:  0.9696233292831107
allele u-net precision:  0.9921847246891652
allele u-net recall:  0.9480651731160896


## 30 mixtures
* allele u-net f1:  0.9558823529411765
* allele u-net precision:  0.9848484848484849
* allele u-net recall:  0.9285714285714286

## Comapring to Humans

In [10]:
from DNAnet.models.segmentation.human_analysis import HumanAnalysis

human_model = HumanAnalysis()

human_predictions = []
for img in hid_dataset:
    pred = human_model.predict(img)
    human_predictions.append(pred)



In [11]:
from DNAnet.evaluation.segmentation.allele_metrics import allele_f1_score, allele_precision, allele_recall

print("Allele F1:", allele_f1_score(hid_dataset, human_predictions))
print("Allele Precision:", allele_precision(hid_dataset, human_predictions))
print("Allele Recall:", allele_recall(hid_dataset, human_predictions))


Allele F1: 0.9762108505904501
Allele Precision: 0.9844666896789782
Allele Recall: 0.9680923285811269
